# Аудит модели: фиксированные кандидаты и проверка обобщения
Не перезаписывает `models/submission.csv`. Для первого запуска достаточно готового `data_by_dewallen/features.csv` из архива патча. Не запускайте NLP или генерацию расширенных меток перед сравнением.

Проверяем 4 заранее заданных кандидата на одинаковых StratifiedGroupKFold (группы продавцов), с порогом 0.50. Повторная CV на знакомых 300 метках **не является независимой оценкой**. Псевдометки держим отдельно от verified, исключаем продавцов проверочного фолда. Survey не становится истиной; только консервативная слабая разметка при согласовании сигналов. Сравниваем варианты, не выбираем победителя автоматически.

In [1]:
import os
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name in {"analysis", "feature", "models"}:
    repo_root = repo_root.parent

assert (repo_root / "requirements.txt").is_file() and (repo_root / "data/listings.csv").is_file(), (
    "Запустите ноутбук из корня репозитория или из папки, в которой он лежит"
)
os.chdir(repo_root)

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold


In [3]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
EXTENDED_VERIFIED_PATH = Path("analysis/extend_verified.csv")
LISTINGS_PATH = Path("data/listings.csv")
OUTPUT_DIR = Path("models/audit_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
CV_SEEDS = [11, 42, 107]
THRESHOLD = 0.50
CHAT_MODEL_SHARE = 0.75
CHAT_LABEL_WEIGHT = 0.10
CHAT_C = 0.3
VERIFIED_C = 1.0
WEAK_WEIGHT_CAP = 0.03
WEAK_CLASS_MASS = 15.0


In [4]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.is_unique, "Повторяются ID в проверенной разметке"
assert verified.index.isin(features.index).all()
assert verified["is_deal"].isin([0, 1]).all(), "Некорректные метки в verified.csv"
assert features.columns.is_unique

feature_columns = features.columns.difference(["seller_id", "is_deal", "weight"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]
categories = features.loc[y.index, "category"]
assert categories.notna().all(), "Для CatShareMAE нужна категория каждого проверенного объявления"

print(f"Объявлений в матрице: {len(X)}")
print(f"Признаков: {X.shape[1]}")
print(f"Объявлений с проверенными метками: {len(y)}")
assert X.shape == (15274, 139), f"Нужна матрица с 4 interaction-признаками, получено {X.shape}"
assert len(y) == 300
seller_groups = features["seller_id"]
assert seller_groups.notna().all()
blocked_ids = listings.index[listings["close_type"].eq("заблокировано")]


Объявлений в матрице: 15274
Признаков: 139
Объявлений с проверенными метками: 300


In [5]:
interaction_columns = ["strong_chat_signal", "survey_changed_no_strong_chat",
                       "survey_elsewhere_no_strong_chat", "survey_avito_strong_chat"]
legacy_nlp_columns = ["scored_message_count", "strongest_positive_signal", "strongest_negative_signal", "deal_signal_score", "deal_signal_strength"]
assert set(interaction_columns + legacy_nlp_columns).issubset(X.columns)
legacy = pd.read_csv(EXTENDED_VERIFIED_PATH, index_col="listing_id")
assert legacy.index.is_unique
assert legacy.loc[y.index, "is_deal"].eq(y).all()
assert legacy.loc[y.index, "weight"].eq(1).all()
legacy = legacy.loc[legacy["weight"].eq(1) & ~legacy.index.isin(y.index), ["is_deal", "weight"]].assign(weight=CHAT_LABEL_WEIGHT)
assert len(legacy) == 726, f"Ожидали прежние 726 NLP, получили {len(legacy)}"
# Два источника слабой разметки: совместные признаки опроса и переписки.
# Не используем все positive-only survey/NLP как ground truth.
pos = X["survey_avito_strong_chat"].eq(1)
neg = X["survey_changed_no_strong_chat"].eq(1) | X["survey_elsewhere_no_strong_chat"].eq(1)
weak = pd.DataFrame({"is_deal": pos.astype(int)}, index=X.index).loc[pos ^ neg]
weak = weak.loc[~weak.index.isin(y.index) & ~weak.index.isin(blocked_ids)].copy()
assert weak.index.is_unique
assert weak.index.intersection(y.index).empty
assert legacy.index.intersection(y.index).empty
print("Legacy NLP:", legacy["is_deal"].value_counts().to_dict())
print("Weak candidates:", weak["is_deal"].value_counts().to_dict())


Legacy NLP: {1: 726}
Weak candidates: {0: 2454, 1: 1150}


In [6]:
def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, make_column_selector(dtype_include=["number", "bool"])),
    ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=["number", "bool"])),
])
verified_model = make_pipeline(
    preprocessor,
    LogisticRegression(C=VERIFIED_C, max_iter=2000),
)
chat_model = clone(verified_model).set_params(logisticregression__C=CHAT_C)

In [7]:
# Конфигурации зафиксированы до запуска CV.
# Контроль с текущими данными, затем изолированное добавление interactions,
# затем отказ от старого NLP, затем сбалансированная слабая разметка.
CANDIDATES = {
    "01_control": (X.drop(columns=interaction_columns), "legacy"),
    "02_interactions": (X, "legacy"),
    "03_verified_only": (X.drop(columns=legacy_nlp_columns), "none"),
    "04_balanced_weak": (X.drop(columns=legacy_nlp_columns), "weak"),
}

def pseudo_for(mode, heldout_sellers):
    if mode == "none":
        return legacy.iloc[:0].copy()
    pool = (legacy if mode == "legacy" else weak).copy()
    pool = pool.loc[~seller_groups.loc[pool.index].isin(heldout_sellers)]
    if mode == "weak":
        counts = pool["is_deal"].value_counts()
        if len(counts) != 2:
            return legacy.iloc[:0].copy()
        # Одинаковый суммарный вес каждого класса, ограничиваем вес одной метки.
        mass = min(WEAK_CLASS_MASS, WEAK_WEIGHT_CAP * counts.min())
        pool["weight"] = pool["is_deal"].map(mass / counts)
    assert pool.index.intersection(y.index).empty
    return pool


def fit_one(estimator, frame, target, pseudo):
    labels = pd.concat([target.to_frame().assign(weight=1.0), pseudo])
    assert labels.index.is_unique
    return clone(estimator).fit(frame.loc[labels.index], labels["is_deal"],
        logisticregression__sample_weight=labels["weight"])


def predict(frame, mode, target, test_ids, heldout_sellers):
    empty = legacy.iloc[:0]
    gold = fit_one(verified_model, frame, target, empty)
    p = gold.predict_proba(frame.loc[test_ids])[:, 1]
    if mode != "none":
        pseudo = pseudo_for(mode, heldout_sellers)
        augmented = fit_one(chat_model, frame, target, pseudo)
        p = CHAT_MODEL_SHARE * augmented.predict_proba(frame.loc[test_ids])[:, 1] + (1 - CHAT_MODEL_SHARE) * p
    p[pd.Index(test_ids).isin(blocked_ids)] = 0.0
    return p


In [8]:
def competition_score(f1, cat_share_mae):
    f1_points = 20 * np.clip((f1 - 0.60) / 0.25, 0, 1)
    share_points = 10 * np.clip((15 - cat_share_mae) / 14, 0, 1)
    return f1_points + share_points


In [9]:
# Общие разбиения: ни один продавец из gold test не входит в gold train или псевдометки.
results = []
oof_records = []
for seed in CV_SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    folds = list(cv.split(X.loc[y.index], y, groups=seller_groups.loc[y.index]))
    for name, (frame, mode) in CANDIDATES.items():
        oof = pd.Series(np.nan, index=y.index)
        for train, test in folds:
            target = y.iloc[train]
            test_ids = y.index[test]
            heldout = set(seller_groups.loc[test_ids])
            assert not set(seller_groups.loc[target.index]) & heldout
            oof.loc[test_ids] = predict(frame, mode, target, test_ids, heldout)
        assert oof.notna().all()
        pred = (oof > THRESHOLD).astype(int)
        shares = pd.DataFrame({"actual": y, "pred": pred, "category": categories}).groupby("category")[["actual", "pred"]].mean()
        mae = 100 * (shares["actual"] - shares["pred"]).abs().mean()
        f1 = f1_score(y, pred, zero_division=0)
        results.append({"candidate": name, "seed": seed, "F1": f1, "CatShareMAE": mae,
                        "Precision": precision_score(y, pred, zero_division=0), "Recall": recall_score(y, pred, zero_division=0), "Score": competition_score(f1, mae)})
        oof_records.append(pd.DataFrame({"candidate": name, "seed": seed, "listing_id": y.index, "actual": y.to_numpy(), "probability": oof.to_numpy(), "prediction": pred.to_numpy()}))
        print(name, seed, f"F1={f1:.4f} Score={competition_score(f1, mae):.4f}", flush=True)
results = pd.DataFrame(results)
display(results.round(4))
display(results.groupby("candidate").mean(numeric_only=True).drop(columns="seed").round(4))
results.to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
pd.concat(oof_records, ignore_index=True).to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)


01_control 11 F1=0.8326 Score=26.5373
02_interactions 11 F1=0.8326 Score=26.5373
03_verified_only 11 F1=0.8036 Score=24.0977
04_balanced_weak 11 F1=0.8295 Score=25.2482
01_control 42 F1=0.8151 Score=24.6440
02_interactions 42 F1=0.8220 Score=25.3262
03_verified_only 42 F1=0.8190 Score=24.8788
04_balanced_weak 42 F1=0.8348 Score=25.7579
01_control 107 F1=0.8270 Score=25.9050
02_interactions 107 F1=0.8305 Score=26.4121
03_verified_only 107 F1=0.8070 Score=25.4940
04_balanced_weak 107 F1=0.8230 Score=25.6557


,candidate,seed,F1,CatShareMAE,Precision,Recall,Score
0,01_control,11,0.8326,3.9009,0.8291,0.8362,26.5373
1,02_interactions,11,0.8326,3.9009,0.8291,0.8362,26.5373
2,03_verified_only,11,0.8036,4.0632,0.8333,0.7759,24.0977
3,04_balanced_weak,11,0.8295,5.3557,0.8911,0.7759,25.2482
4,01_control,42,0.8151,4.5925,0.7951,0.8362,24.6440
5,02_interactions,42,0.8220,4.4111,0.8083,0.8362,25.3262
6,03_verified_only,42,0.8190,4.6939,0.8190,0.8190,24.8788
7,04_balanced_weak,42,0.8348,5.2346,0.8421,0.8276,25.7579
8,01_control,107,0.8270,4.1575,0.8099,0.8448,25.9050
9,02_interactions,107,0.8305,3.8400,0.8167,0.8448,26.4121


,F1,CatShareMAE,Precision,Recall,Score
candidate,,,,,
01_control,0.8249,4.2170,0.8114,0.8391,25.6954
02_interactions,0.8284,4.0507,0.8180,0.8391,26.0919
03_verified_only,0.8099,3.7505,0.8246,0.7960,24.8235
04_balanced_weak,0.8291,4.8831,0.8595,0.8017,25.5540


In [10]:
# Сохраняем отдельные кандидаты. Никакого автоматического выбора по тем же 300 меткам.
for name, (frame, mode) in CANDIDATES.items():
    p = predict(frame, mode, y, frame.index, set())
    submission = pd.DataFrame({"listing_id": frame.index, "is_deal": (p > THRESHOLD).astype(int)})
    assert submission["listing_id"].is_unique
    assert set(submission["listing_id"]) == set(listings.index)
    assert submission["is_deal"].isin([0, 1]).all()
    assert not submission.loc[submission["listing_id"].isin(blocked_ids), "is_deal"].any()
    path = OUTPUT_DIR / f"submission_{name}.csv"
    submission.to_csv(path, index=False)
    print(path, "positive:", submission["is_deal"].sum())


models\audit_outputs\submission_01_control.csv positive: 6011
models\audit_outputs\submission_02_interactions.csv positive: 6019
models\audit_outputs\submission_03_verified_only.csv positive: 5673
models\audit_outputs\submission_04_balanced_weak.csv positive: 5538
